# Model B (depthwise separable CNN): Colab T4 configuration sweep

Runs Model B the same way Model A was run in `results/model_a/model_a_configurations.ipynb`,
on the same Colab T4 GPU, so the two models' timings are comparable.

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save. Then run the cells from top to bottom.

Differences from the Model A notebook, to follow the assignment rules:
- Configurations are ranked by **validation accuracy**. Test accuracy is shown for reference only and is never used to choose.
- A run whose results already exist in Drive is skipped, so if Colab disconnects you can rerun everything and it continues where it stopped.

## 1. Setup

In [1]:
# Mount Google Drive so results survive a Colab disconnect.
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
DRIVE_ROOT = Path("/content/drive/MyDrive/pattern_cnn_experiments/model_b")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Runs are saved under:", DRIVE_ROOT)

Mounted at /content/drive
Runs are saved under: /content/drive/MyDrive/pattern_cnn_experiments/model_b


In [2]:
# Clone the repository (first time) or update it (later runs).
# Use "main" here once Model B's branch has been merged.
BRANCH = "feature/model-b-colab"
REPO = Path("/content/CNN-for-Image-Classification")

%cd /content
if not REPO.exists():
    !git clone https://github.com/Seth724/CNN-for-Image-Classification.git
%cd /content/CNN-for-Image-Classification
!git fetch origin
!git checkout {BRANCH}
!git pull origin {BRANCH}
!git log --oneline -1

/content
Cloning into 'CNN-for-Image-Classification'...
remote: Enumerating objects: 279, done.
remote: Counting objects: 100% (108/108), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 279 (delta 32), reused 70 (delta 22), pack-reused 171 (from 1)
Receiving objects: 100% (279/279), 32.02 MiB | 33.77 MiB/s, done.
Resolving deltas: 100% (95/95), done.
/content/CNN-for-Image-Classification
branch 'feature/model-b-colab' set up to track 'origin/feature/model-b-colab'.
Switched to a new branch 'feature/model-b-colab'
From https://github.com/Seth724/CNN-for-Image-Classification
 * branch            feature/model-b-colab -> FETCH_HEAD
Already up to date.
3cfd0e0 (HEAD -> feature/model-b-colab, origin/feature/model-b-colab) Add Colab T4 notebook for the Model B configuration sweep


In [3]:
import tensorflow as tf
print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU devices:", gpus)
assert gpus, "No GPU found. Runtime > Change runtime type > T4 GPU, then rerun from the top."
!nvidia-smi --query-gpu=name --format=csv,noheader

TensorFlow: 2.21.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Tesla T4


In [4]:
# The processed 70/15/15 split is in the repository; this only checks it.
!python scripts/verify_data.py

Verification PASSED
Dataset : Fashion-MNIST
Seed    : 42
Train   : (49000, 28, 28)
Val     : (10500, 28, 28)
Test    : (10500, 28, 28)


## 2. Helpers
`run_b(...)` trains one configuration into its own Drive folder and skips it if it already finished.
`summary()` collects every finished run into one table, ranked by final validation accuracy.

In [5]:
import json
import subprocess
from pathlib import Path
import pandas as pd

def run_name(optimizer, lr, batch_size, epochs, momentum=None):
    name = f"model_b_{optimizer}_lr{lr:g}_bs{batch_size}_e{epochs}"
    return name + (f"_m{momentum:g}" if momentum is not None else "")

def run_b(optimizer, lr, batch_size=128, epochs=20, momentum=None, seed=42):
    name = run_name(optimizer, lr, batch_size, epochs, momentum)
    out = DRIVE_ROOT / name
    if (out / "metrics" / "evaluation.json").exists():
        print(f"skip  {name} (already finished)")
        return name
    cmd = (f"python scripts/train_model_b.py --epochs {epochs} --batch-size {batch_size} "
           f"--optimizer {optimizer} --learning-rate {lr} --seed {seed} --output-dir '{out}'")
    if momentum is not None:
        cmd += f" --momentum {momentum}"
    print("run  ", name)
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    # Show only the summary lines; the full log is saved next to the results.
    (out).mkdir(parents=True, exist_ok=True)
    (out / "train_log.txt").write_text(result.stdout + result.stderr)
    for line in result.stdout.splitlines():
        if line.startswith(("Training time", "Final validation", "Test accuracy")):
            print("      ", line)
    if result.returncode != 0:
        print(result.stderr[-3000:])
        raise RuntimeError(f"{name} failed; see {out / 'train_log.txt'}")
    return name

def summary():
    rows = []
    for path in sorted(DRIVE_ROOT.glob("model_b_*/metrics/evaluation.json")):
        m = json.loads(path.read_text())
        rows.append({
            "run": path.parent.parent.name,
            "optimizer": m["optimizer"],
            "learning_rate": m["learning_rate"],
            "momentum": m["momentum"],
            "batch_size": m["batch_size"],
            "epochs": m["epochs"],
            "final_val_accuracy": round(m["final_val_accuracy"], 4),
            "best_val_accuracy": round(m["best_val_accuracy"], 4),
            "mean_epoch_time_s": m["mean_epoch_time_s"],
            "test_accuracy (reference only)": round(m["test_accuracy"], 4),
        })
    return pd.DataFrame(rows).sort_values("final_val_accuracy", ascending=False).reset_index(drop=True)

## 3. Smoke test
One epoch, to check the code, data and GPU work before the real runs. Accuracy will be low; that is expected.

In [6]:
!python scripts/train_model_b.py --epochs 1 --output-dir /content/smoke_test_b
!ls -R /content/smoke_test_b | head -20

W0000 00:00:1792246538.226550    1137 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
Model: "model_b"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image (InputLayer)              │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1_relu (Conv2D)             │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool1 (MaxPooling2D)            │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sepconv2_relu (SeparableConv2D) │ (None, 14, 14, 64)     │         2,400 │
├───────────────────

## 4. Stage 1: optimizers (Q3)
Same four runs as Model A: 20 epochs, batch 128, default learning rates.

In [7]:
run_b("adam", 0.001)
run_b("rmsprop", 0.001)
run_b("sgd", 0.01)
run_b("sgd_momentum", 0.01, momentum=0.9)
summary()

run   model_b_adam_lr0.001_bs128_e20
       Training time: 50.22 seconds
       Final validation accuracy: 0.8574
       Test accuracy: 0.8650
run   model_b_rmsprop_lr0.001_bs128_e20
       Training time: 57.13 seconds
       Final validation accuracy: 0.8476
       Test accuracy: 0.8554
run   model_b_sgd_lr0.01_bs128_e20
       Training time: 46.94 seconds
       Final validation accuracy: 0.2732
       Test accuracy: 0.2647
run   model_b_sgd_momentum_lr0.01_bs128_e20_m0.9
       Training time: 47.89 seconds
       Final validation accuracy: 0.8117
       Test accuracy: 0.8200


,run,optimizer,learning_rate,momentum,batch_size,epochs,final_val_accuracy,best_val_accuracy,mean_epoch_time_s,test_accuracy (reference only)
0,model_b_adam_lr0.001_bs128_e20,adam,0.001,0.0,128,20,0.8574,0.8574,1.878,0.8650
1,model_b_rmsprop_lr0.001_bs128_e20,rmsprop,0.001,0.0,128,20,0.8476,0.8476,2.300,0.8554
2,model_b_sgd_momentum_lr0.01_bs128_e20_m0.9,sgd_momentum,0.010,0.9,128,20,0.8117,0.8117,1.843,0.8200
3,model_b_sgd_lr0.01_bs128_e20,sgd,0.010,0.0,128,20,0.2732,0.2732,1.803,0.2647


In [8]:
# Optional, for Q3 ("impact of the momentum parameter"): SGD with momentum 0.5 and 0.99.
run_b("sgd_momentum", 0.01, momentum=0.5)
run_b("sgd_momentum", 0.01, momentum=0.99)
summary()[lambda d: d.optimizer.str.startswith("sgd")]

run   model_b_sgd_momentum_lr0.01_bs128_e20_m0.5
       Training time: 47.86 seconds
       Final validation accuracy: 0.6857
       Test accuracy: 0.6888
run   model_b_sgd_momentum_lr0.01_bs128_e20_m0.99
       Training time: 47.94 seconds
       Final validation accuracy: 0.8791
       Test accuracy: 0.8867


,run,optimizer,learning_rate,momentum,batch_size,epochs,final_val_accuracy,best_val_accuracy,mean_epoch_time_s,test_accuracy (reference only)
0,model_b_sgd_momentum_lr0.01_bs128_e20_m0.99,sgd_momentum,0.01,0.99,128,20,0.8791,0.8820,1.845,0.8867
3,model_b_sgd_momentum_lr0.01_bs128_e20_m0.9,sgd_momentum,0.01,0.90,128,20,0.8117,0.8117,1.843,0.8200
4,model_b_sgd_momentum_lr0.01_bs128_e20_m0.5,sgd_momentum,0.01,0.50,128,20,0.6857,0.6857,1.833,0.6888
5,model_b_sgd_lr0.01_bs128_e20,sgd,0.01,0.00,128,20,0.2732,0.2732,1.803,0.2647


## 5. Stage 2: learning rate
The same learning rates Model A tried, for the two best optimizers from stage 1 (by validation accuracy).

In [9]:
stage1 = summary()
stage1 = stage1[(stage1.epochs == 20) & (stage1.batch_size == 128) & (stage1.momentum.isin([0.0, 0.9]))]
TOP_OPTIMIZERS = list(stage1.drop_duplicates("optimizer").optimizer[:2])
print("Best two optimizers on validation:", TOP_OPTIMIZERS)

for optimizer in TOP_OPTIMIZERS:
    for lr in [0.0003, 0.001, 0.002, 0.003, 0.004]:
        if optimizer.startswith("sgd"):
            lr = lr * 10   # SGD needs ~10x larger steps than adaptive optimizers
        run_b(optimizer, lr, momentum=0.9 if optimizer == "sgd_momentum" else None)
summary().head(10)

Best two optimizers on validation: ['adam', 'rmsprop']
run   model_b_adam_lr0.0003_bs128_e20
       Training time: 49.58 seconds
       Final validation accuracy: 0.7828
       Test accuracy: 0.7905
skip  model_b_adam_lr0.001_bs128_e20 (already finished)
run   model_b_adam_lr0.002_bs128_e20
       Training time: 50.07 seconds
       Final validation accuracy: 0.8740
       Test accuracy: 0.8825
run   model_b_adam_lr0.003_bs128_e20
       Training time: 51.41 seconds
       Final validation accuracy: 0.8741
       Test accuracy: 0.8865
run   model_b_adam_lr0.004_bs128_e20
       Training time: 51.41 seconds
       Final validation accuracy: 0.8910
       Test accuracy: 0.8952
run   model_b_rmsprop_lr0.0003_bs128_e20
       Training time: 49.99 seconds
       Final validation accuracy: 0.7706
       Test accuracy: 0.7715
skip  model_b_rmsprop_lr0.001_bs128_e20 (already finished)
run   model_b_rmsprop_lr0.002_bs128_e20
       Training time: 48.42 seconds
       Final validation accuracy: 

,run,optimizer,learning_rate,momentum,batch_size,epochs,final_val_accuracy,best_val_accuracy,mean_epoch_time_s,test_accuracy (reference only)
0,model_b_rmsprop_lr0.003_bs128_e20,rmsprop,0.003,0.00,128,20,0.8913,0.8932,1.853,0.9001
1,model_b_adam_lr0.004_bs128_e20,adam,0.004,0.00,128,20,0.8910,0.8951,1.927,0.8952
2,model_b_rmsprop_lr0.004_bs128_e20,rmsprop,0.004,0.00,128,20,0.8856,0.9005,1.840,0.8931
3,model_b_sgd_momentum_lr0.01_bs128_e20_m0.99,sgd_momentum,0.010,0.99,128,20,0.8791,0.8820,1.845,0.8867
4,model_b_rmsprop_lr0.002_bs128_e20,rmsprop,0.002,0.00,128,20,0.8790,0.8816,1.837,0.8881
5,model_b_adam_lr0.003_bs128_e20,adam,0.003,0.00,128,20,0.8741,0.8869,1.961,0.8865
6,model_b_adam_lr0.002_bs128_e20,adam,0.002,0.00,128,20,0.8740,0.8781,1.880,0.8825
7,model_b_adam_lr0.001_bs128_e20,adam,0.001,0.00,128,20,0.8574,0.8574,1.878,0.8650
8,model_b_rmsprop_lr0.001_bs128_e20,rmsprop,0.001,0.00,128,20,0.8476,0.8476,2.300,0.8554
9,model_b_sgd_momentum_lr0.01_bs128_e20_m0.9,sgd_momentum,0.010,0.90,128,20,0.8117,0.8117,1.843,0.8200


## 6. Stage 3: batch size
Batch sizes 64 and 256 around the best stage-2 configuration.

In [10]:
def best(df):
    return df.iloc[0]

def momentum_of(row):
    return float(row.momentum) if row.optimizer == "sgd_momentum" else None

b = best(summary()[lambda d: (d.epochs == 20)])
print("Best so far:", b.run)
for batch_size in [64, 256]:
    run_b(b.optimizer, b.learning_rate, batch_size=batch_size, momentum=momentum_of(b))
summary().head(10)

Best so far: model_b_rmsprop_lr0.003_bs128_e20
run   model_b_rmsprop_lr0.003_bs64_e20
       Training time: 66.09 seconds
       Final validation accuracy: 0.8747
       Test accuracy: 0.8866
run   model_b_rmsprop_lr0.003_bs256_e20
       Training time: 40.15 seconds
       Final validation accuracy: 0.8713
       Test accuracy: 0.8807


,run,optimizer,learning_rate,momentum,batch_size,epochs,final_val_accuracy,best_val_accuracy,mean_epoch_time_s,test_accuracy (reference only)
0,model_b_rmsprop_lr0.003_bs128_e20,rmsprop,0.003,0.00,128,20,0.8913,0.8932,1.853,0.9001
1,model_b_adam_lr0.004_bs128_e20,adam,0.004,0.00,128,20,0.8910,0.8951,1.927,0.8952
2,model_b_rmsprop_lr0.004_bs128_e20,rmsprop,0.004,0.00,128,20,0.8856,0.9005,1.840,0.8931
3,model_b_sgd_momentum_lr0.01_bs128_e20_m0.99,sgd_momentum,0.010,0.99,128,20,0.8791,0.8820,1.845,0.8867
4,model_b_rmsprop_lr0.002_bs128_e20,rmsprop,0.002,0.00,128,20,0.8790,0.8816,1.837,0.8881
5,model_b_rmsprop_lr0.003_bs64_e20,rmsprop,0.003,0.00,64,20,0.8747,0.8936,2.732,0.8866
6,model_b_adam_lr0.003_bs128_e20,adam,0.003,0.00,128,20,0.8741,0.8869,1.961,0.8865
7,model_b_adam_lr0.002_bs128_e20,adam,0.002,0.00,128,20,0.8740,0.8781,1.880,0.8825
8,model_b_rmsprop_lr0.003_bs256_e20,rmsprop,0.003,0.00,256,20,0.8713,0.8713,1.419,0.8807
9,model_b_adam_lr0.001_bs128_e20,adam,0.001,0.00,128,20,0.8574,0.8574,1.878,0.8650


## 7. Stage 4: epochs
Longer training for the best configuration. 20 is the assignment minimum, so nothing below 20 is tried.

In [11]:
b = best(summary()[lambda d: (d.epochs == 20)])
print("Best 20-epoch configuration:", b.run)
for epochs in [30, 40, 60]:
    run_b(b.optimizer, b.learning_rate, batch_size=int(b.batch_size), epochs=epochs, momentum=momentum_of(b))
summary().head(10)

Best 20-epoch configuration: model_b_rmsprop_lr0.003_bs128_e20
run   model_b_rmsprop_lr0.003_bs128_e30
       Training time: 66.81 seconds
       Final validation accuracy: 0.8956
       Test accuracy: 0.9021
run   model_b_rmsprop_lr0.003_bs128_e40
       Training time: 86.23 seconds
       Final validation accuracy: 0.9009
       Test accuracy: 0.9033
run   model_b_rmsprop_lr0.003_bs128_e60
       Training time: 122.15 seconds
       Final validation accuracy: 0.9038
       Test accuracy: 0.9083


,run,optimizer,learning_rate,momentum,batch_size,epochs,final_val_accuracy,best_val_accuracy,mean_epoch_time_s,test_accuracy (reference only)
0,model_b_rmsprop_lr0.003_bs128_e60,rmsprop,0.003,0.00,128,60,0.9038,0.9155,1.844,0.9083
1,model_b_rmsprop_lr0.003_bs128_e40,rmsprop,0.003,0.00,128,40,0.9009,0.9121,1.865,0.9033
2,model_b_rmsprop_lr0.003_bs128_e30,rmsprop,0.003,0.00,128,30,0.8956,0.9080,1.841,0.9021
3,model_b_rmsprop_lr0.003_bs128_e20,rmsprop,0.003,0.00,128,20,0.8913,0.8932,1.853,0.9001
4,model_b_adam_lr0.004_bs128_e20,adam,0.004,0.00,128,20,0.8910,0.8951,1.927,0.8952
5,model_b_rmsprop_lr0.004_bs128_e20,rmsprop,0.004,0.00,128,20,0.8856,0.9005,1.840,0.8931
6,model_b_sgd_momentum_lr0.01_bs128_e20_m0.99,sgd_momentum,0.010,0.99,128,20,0.8791,0.8820,1.845,0.8867
7,model_b_rmsprop_lr0.002_bs128_e20,rmsprop,0.002,0.00,128,20,0.8790,0.8816,1.837,0.8881
8,model_b_rmsprop_lr0.003_bs64_e20,rmsprop,0.003,0.00,64,20,0.8747,0.8936,2.732,0.8866
9,model_b_adam_lr0.003_bs128_e20,adam,0.003,0.00,128,20,0.8741,0.8869,1.961,0.8865


## 8. Choose the final configurations
Model A reported its best Adam and best RMSprop runs, so Model B does the same: the best run per optimizer, by validation accuracy.

In [12]:
everything = summary()
SELECTED = {}
for optimizer in ["adam", "rmsprop"]:
    rows = everything[everything.optimizer == optimizer]
    if len(rows):
        SELECTED[f"model_b_{optimizer}"] = rows.iloc[0].run
if everything.iloc[0].run not in SELECTED.values():
    SELECTED["model_b_best"] = everything.iloc[0].run   # overall best if it is neither Adam nor RMSprop
print(SELECTED)
everything.to_csv(DRIVE_ROOT / "model_b_all_runs.csv", index=False)
everything

{'model_b_adam': 'model_b_adam_lr0.004_bs128_e20', 'model_b_rmsprop': 'model_b_rmsprop_lr0.003_bs128_e60'}


,run,optimizer,learning_rate,momentum,batch_size,epochs,final_val_accuracy,best_val_accuracy,mean_epoch_time_s,test_accuracy (reference only)
0,model_b_rmsprop_lr0.003_bs128_e60,rmsprop,0.0030,0.00,128,60,0.9038,0.9155,1.844,0.9083
1,model_b_rmsprop_lr0.003_bs128_e40,rmsprop,0.0030,0.00,128,40,0.9009,0.9121,1.865,0.9033
2,model_b_rmsprop_lr0.003_bs128_e30,rmsprop,0.0030,0.00,128,30,0.8956,0.9080,1.841,0.9021
3,model_b_rmsprop_lr0.003_bs128_e20,rmsprop,0.0030,0.00,128,20,0.8913,0.8932,1.853,0.9001
4,model_b_adam_lr0.004_bs128_e20,adam,0.0040,0.00,128,20,0.8910,0.8951,1.927,0.8952
5,model_b_rmsprop_lr0.004_bs128_e20,rmsprop,0.0040,0.00,128,20,0.8856,0.9005,1.840,0.8931
6,model_b_sgd_momentum_lr0.01_bs128_e20_m0.99,sgd_momentum,0.0100,0.99,128,20,0.8791,0.8820,1.845,0.8867
7,model_b_rmsprop_lr0.002_bs128_e20,rmsprop,0.0020,0.00,128,20,0.8790,0.8816,1.837,0.8881
8,model_b_rmsprop_lr0.003_bs64_e20,rmsprop,0.0030,0.00,64,20,0.8747,0.8936,2.732,0.8866
9,model_b_adam_lr0.003_bs128_e20,adam,0.0030,0.00,128,20,0.8741,0.8869,1.961,0.8865


## 9. Report figures (same four figures as Model A)

In [13]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

for label, run in SELECTED.items():
    run_dir = DRIVE_ROOT / run
    report_dir = run_dir / "report_figures"
    report_dir.mkdir(parents=True, exist_ok=True)
    history = pd.read_csv(run_dir / "metrics" / "history.csv")
    metrics = json.loads((run_dir / "metrics" / "evaluation.json").read_text())

    for key, ylabel, fname in [("loss", "Loss", "loss_curves.png"), ("accuracy", "Accuracy", "accuracy_curves.png")]:
        plt.figure(figsize=(8, 5))
        plt.plot(history["epoch"], history[key], label=f"Training {ylabel.lower()}", linewidth=2)
        plt.plot(history["epoch"], history[f"val_{key}"], label=f"Validation {ylabel.lower()}", linewidth=2)
        plt.xlabel("Epoch"); plt.ylabel(ylabel); plt.title(f"{run}: {ylabel.lower()} curves")
        plt.grid(alpha=0.3); plt.legend(); plt.tight_layout()
        plt.savefig(report_dir / fname, dpi=300); plt.close()

    plt.figure(figsize=(9, 7))
    sns.heatmap(np.array(metrics["confusion_matrix"]), annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted label"); plt.ylabel("True label"); plt.title(f"{run}: test confusion matrix")
    plt.xticks(rotation=45, ha="right"); plt.yticks(rotation=0); plt.tight_layout()
    plt.savefig(report_dir / "confusion_matrix.png", dpi=300); plt.close()

    per_class = pd.DataFrame(metrics["per_class"]); per_class["class"] = class_names
    per_class.set_index("class")[["precision", "recall", "f1_score"]].plot(kind="bar", figsize=(12, 6), ylim=(0, 1))
    plt.xlabel("Class"); plt.ylabel("Score"); plt.title(f"{run}: per-class metrics")
    plt.xticks(rotation=45, ha="right"); plt.grid(axis="y", alpha=0.3); plt.tight_layout()
    plt.savefig(report_dir / "per_class_metrics.png", dpi=300); plt.close()
    print("Created figures:", report_dir)

Created figures: /content/drive/MyDrive/pattern_cnn_experiments/model_b/model_b_adam_lr0.004_bs128_e20/report_figures
Created figures: /content/drive/MyDrive/pattern_cnn_experiments/model_b/model_b_rmsprop_lr0.003_bs128_e60/report_figures


## 10. Comparison table (same columns as `results/model_a/model_a_comparison.csv`)

In [14]:
rows = []
for label, run in SELECTED.items():
    m = json.loads((DRIVE_ROOT / run / "metrics" / "evaluation.json").read_text())
    rows.append({
        "model": "Model B", "run": run, "optimizer": m["optimizer"], "learning_rate": m["learning_rate"],
        "epochs": m["epochs"], "batch_size": m["batch_size"], "total_parameters": m["total_parameters"],
        "model_size_kb": m["model_size_kb"], "training_time_per_epoch_s": m["mean_epoch_time_s"],
        "test_accuracy": m["test_accuracy"], "macro_precision": m["macro_precision"],
        "macro_recall": m["macro_recall"], "macro_f1": m["macro_f1"],
        "final_val_accuracy": m["final_val_accuracy"], "total_macs": m["total_macs"],
        "cpu_latency_ms": m["cpu_latency_ms"],
    })
model_b_comparison = pd.DataFrame(rows).sort_values("final_val_accuracy", ascending=False)
model_b_comparison.to_csv(DRIVE_ROOT / "model_b_comparison.csv", index=False)
model_b_comparison

,model,run,optimizer,learning_rate,epochs,batch_size,total_parameters,model_size_kb,training_time_per_epoch_s,test_accuracy,macro_precision,macro_recall,macro_f1,final_val_accuracy,total_macs,cpu_latency_ms
1,Model B,model_b_rmsprop_lr0.003_bs128_e60,rmsprop,0.003,60,128,20522,203.68,1.844,0.908286,0.909589,0.908286,0.906928,0.903810,1122112,0.898
0,Model B,model_b_adam_lr0.004_bs128_e20,adam,0.004,20,128,20522,288.33,1.927,0.895238,0.898030,0.895238,0.892420,0.891048,1122112,0.894


## 11. Package the results and download them
This copies the small files (no `.keras` weights) into the same layout as `results/model_a/` and downloads one zip.
Send that zip back (or put it in the repo folder) and it gets committed.

In [15]:
import shutil

package = Path("/content/model_b_results/results/model_b")
if package.exists():
    shutil.rmtree(package)
package.mkdir(parents=True)

for label, run in SELECTED.items():
    src, dst = DRIVE_ROOT / run, package / label
    dst.mkdir()
    for name in ["architecture.json", "evaluation.json", "history.csv"]:
        shutil.copy(src / "metrics" / name, dst / name)
    shutil.copytree(src / "report_figures", dst / "report_figures")
    (dst / "run_name.txt").write_text(run + "\n")

for name in ["model_b_comparison.csv", "model_b_all_runs.csv"]:
    shutil.copy(DRIVE_ROOT / name, package / name)

archive = shutil.make_archive(str(DRIVE_ROOT / "model_b_report_artifacts"), "zip", "/content/model_b_results")
!find /content/model_b_results -type f | sort
print("Zip saved to Drive:", archive)

from google.colab import files
files.download(archive)

/content/model_b_results/results/model_b/model_b_adam/architecture.json
/content/model_b_results/results/model_b/model_b_adam/evaluation.json
/content/model_b_results/results/model_b/model_b_adam/history.csv
/content/model_b_results/results/model_b/model_b_adam/report_figures/accuracy_curves.png
/content/model_b_results/results/model_b/model_b_adam/report_figures/confusion_matrix.png
/content/model_b_results/results/model_b/model_b_adam/report_figures/loss_curves.png
/content/model_b_results/results/model_b/model_b_adam/report_figures/per_class_metrics.png
/content/model_b_results/results/model_b/model_b_adam/run_name.txt
/content/model_b_results/results/model_b/model_b_all_runs.csv
/content/model_b_results/results/model_b/model_b_comparison.csv
/content/model_b_results/results/model_b/model_b_rmsprop/architecture.json
/content/model_b_results/results/model_b/model_b_rmsprop/evaluation.json
/content/model_b_results/results/model_b/model_b_rmsprop/history.csv
/content/model_b_results/re

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>